# Speech-to-Text (STT) — transcription benchmark on Apple Silicon

Transcribes every clip of an audio set with a model in **float16** on MLX and records, per clip: audio length, input /
output tokens, output words, first token, time to first token, inference rate and inference time, plus system metrics
(CPU / GPU / NPU utilisation, power, temperature) sampled by `macmon`. The summary adds the word error rate against
each clip's exact reference text.

Whisper runs with [`mlx-whisper`](https://github.com/ml-explore/mlx-examples/tree/main/whisper), Parakeet with
[`parakeet-mlx`](https://github.com/senstella/parakeet-mlx). Each model is kept as a local float16 copy at
`models/<model>-fp16/` (made once from its MLX checkpoint on Hugging Face) and loaded from there.

**Outputs**
- `outputs/stt/<family>/<params>/<audio set>/`: one transcript per clip and `<model>_simple_metrics.json`
- `metrics_analysis/stt/<family>/<params>/<audio set>/`: `Model_Level_Metrics/<model>_clip_metrics.csv`
  and `System_Level_Metrics/<clip id>_system_metrics.csv`

## 1. Settings and model registry

In [ ]:
import gc
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import mlx.core as mx
import pandas as pd
import soundfile as sf

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'prompts').is_dir())   # project root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault('HF_HUB_DISABLE_IMPLICIT_TOKEN', '1')   # every model here is public

# family        size      params  model                    MLX checkpoint on Hugging Face           params (M)  library
REGISTRY = [
    ('Whisper',    'SMALL',  '241M', 'whisper-small',         'mlx-community/whisper-small-mlx',        241,  'whisper'),
    ('Whisper',    'MEDIUM', '762M', 'whisper-medium',        'mlx-community/whisper-medium-mlx',       762,  'whisper'),
    ('Whisper',    'LARGE',  '1.5B', 'whisper-large-v3',      'mlx-community/whisper-large-v3-mlx',     1542, 'whisper'),
    ('Whisper.en', 'SMALL',  '72M',  'whisper-base.en',       'mlx-community/whisper-base.en-mlx',      72,   'whisper'),
    ('Whisper.en', 'MEDIUM', '241M', 'whisper-small.en',      'mlx-community/whisper-small.en-mlx',     241,  'whisper'),
    ('Whisper.en', 'LARGE',  '762M', 'whisper-medium.en',     'mlx-community/whisper-medium.en-mlx',    762,  'whisper'),
    ('Parakeet',   'SMALL',  '115M', 'parakeet-tdt_ctc-110m', 'mlx-community/parakeet-tdt_ctc-110m',    115,  'parakeet'),
    ('Parakeet',   'MEDIUM', '618M', 'parakeet-tdt-0.6b-v2',  'mlx-community/parakeet-tdt-0.6b-v2',     618,  'parakeet'),
    ('Parakeet',   'LARGE',  '1.1B', 'parakeet-tdt-1.1b',     'mlx-community/parakeet-tdt-1.1b',        1071, 'parakeet'),
]
MODELS = pd.DataFrame(REGISTRY, columns=['family', 'size', 'params', 'model', 'source', 'params_m', 'library']).set_index('model')
MODELS['path'] = 'models/' + MODELS.index + '-fp16'   # local float16 copy of each model

# Pick the model here, or from the terminal with STT_FAMILY / STT_SIZE / STT_AUDIO / STT_COMPARE
FAMILY = os.environ.get('STT_FAMILY', 'Whisper')   # Whisper | Whisper.en | Parakeet
SIZE = os.environ.get('STT_SIZE', 'LARGE')         # SMALL | MEDIUM | LARGE
MODEL = MODELS.query('family == @FAMILY and size == @SIZE').index[0]
AUDIO_SET = os.environ.get('STT_AUDIO', 'stt_prompts_10')   # prompts/audio_stt_prompts/<set>.csv (id, audio, voice, reference)
COMPARE_FAMILY = os.environ.get('STT_COMPARE', '1') == '1'  # section 5: also run the family's other models
PARAKEET_CHUNK_S = 120   # Parakeet transcribes audio longer than this in overlapping chunks

print(f'Selected: {FAMILY} / {SIZE} -> {MODEL} -> {MODELS.path[MODEL]}')
MODELS[MODELS.family == FAMILY][['size', 'params', 'library', 'source', 'path']]

## 2. Prepare and load a model

The first time, the model's MLX checkpoint is downloaded and its weights are saved as float16 in `models/<model>-fp16/`
(one-time step); after that it loads from there.

In [ ]:
import mlx_whisper
from huggingface_hub import snapshot_download
from mlx_whisper.transcribe import ModelHolder
from parakeet_mlx import from_pretrained

def fp16_copy(src, dst):
    # Copy a model folder, casting the floating-point tensors of its weight files to float16
    for f in Path(src).rglob('*'):
        out = Path(dst) / f.relative_to(src)
        if f.is_dir():
            continue
        out.parent.mkdir(parents=True, exist_ok=True)
        if f.suffix == '.safetensors' or f.name == 'weights.npz':   # Whisper's weights.npz is saved as weights.safetensors
            w = {k: v.astype(mx.float16) if mx.issubdtype(v.dtype, mx.floating) else v for k, v in mx.load(str(f)).items()}
            mx.save_safetensors(str(out.with_suffix('.safetensors')), w)
        else:
            shutil.copyfile(f, out)

def prepare(name):
    m = MODELS.loc[name]
    if not Path(m.path).exists():
        print(f'Preparing {m.path} from {m.source} (float16), one-time step...')
        try:
            fp16_copy(snapshot_download(m.source, token=False), m.path)
        except Exception:
            shutil.rmtree(m.path, ignore_errors=True)   # don't leave a half-prepared model behind
            raise

def load_model(name):
    m = MODELS.loc[name]
    prepare(name)
    t0 = time.perf_counter()
    if m.library == 'whisper':
        ModelHolder.model, ModelHolder.model_path = None, None   # drop any previously cached Whisper model
        model = ModelHolder.get_model(m.path, mx.float16)        # transcribe() reuses this cached model
    else:
        model = from_pretrained(m.path, dtype=mx.float16)
    mx.eval(model.parameters())
    return model, time.perf_counter() - t0

## 3. Audio set

Each clip comes with the exact text it says (`reference`): the clips were made with macOS `say` in several English
accents.

In [ ]:
CLIPS = pd.read_csv(Path('prompts/audio_stt_prompts') / f'{AUDIO_SET}.csv', dtype={'id': str})
CLIPS['id'] = CLIPS.id.str.zfill(2)
CLIPS['duration'] = [sf.info(p).duration for p in CLIPS.audio]
print(f'{len(CLIPS)} clips, {CLIPS.duration.sum() / 60:.1f} min of audio')
CLIPS[['id', 'voice', 'duration', 'reference']]

## 4. Transcribe with the selected model

Each clip is transcribed separately, after one untimed warm-up clip. Whisper decodes greedily in English with
`condition_on_previous_text=False` (stops long audio drifting into invented text); Parakeet splits audio longer than
`PARAKEET_CHUNK_S` into overlapping chunks. Both return the whole transcript at once, so time to first token equals
inference time. Input tokens are the audio positions the encoder processes: Whisper 1,500 per padded 30 s window,
Parakeet one per 80 ms. Each clip is wrapped in `metrics_monitor_start()` / `metrics_monitor_stop()` (from
`macmon_metrics/`), which saves its system metrics every 100 ms.

In [ ]:
import macmon_metrics.metrics_monitor as mm
from macmon_metrics.metrics_monitor import metrics_monitor_start, metrics_monitor_stop, setup_output_directory
from mlx_whisper.tokenizer import get_tokenizer
from text_metrics import word_errors

if mm._tracker is None:   # start macmon now, so every clip's capture has samples from its first moment
    mm._tracker = mm.MacMonTracker(binary_path=mm._resolve_macmon_binary())
    mm._tracker.start()
    time.sleep(1.5)

def transcribe(model, m, name, audio, duration):
    # Returns (text, token texts, seconds)
    t0 = time.perf_counter()
    if m.library == 'whisper':
        result = mlx_whisper.transcribe(audio, path_or_hf_repo=m.path, language=None if name.endswith('.en') else 'en',
                                        verbose=None, condition_on_previous_text=False, fp16=True)
        total = time.perf_counter() - t0
        tok = get_tokenizer(model.is_multilingual, num_languages=model.num_languages, language='en', task='transcribe')
        tokens = [tok.decode([t]) for s in result['segments'] for t in s['tokens'] if t < tok.eot]   # no timestamps
        return result['text'].strip(), tokens, total
    result = model.transcribe(audio, chunk_duration=PARAKEET_CHUNK_S if duration > PARAKEET_CHUNK_S else None,
                              overlap_duration=15.0)
    return result.text.strip(), [t.text for s in result.sentences for t in s.tokens], time.perf_counter() - t0

def run_model(name):
    # Transcribe every clip with one model; save transcripts, per-clip metrics and a summary
    m = MODELS.loc[name]
    sub = Path(m.family) / m.params / AUDIO_SET
    out_dir = Path('outputs/stt') / sub
    out_dir.mkdir(parents=True, exist_ok=True)
    sys_dir, model_dir = setup_output_directory(str(Path('metrics_analysis/stt') / sub))
    model, load_s = load_model(name)
    transcribe(model, m, name, CLIPS.audio[0], CLIPS.duration[0])   # warm-up
    rows, peak, ref_words, errors = [], 0.0, 0, 0
    for c in CLIPS.itertuples():
        mx.reset_peak_memory()
        metrics_monitor_start(idx=c.id, metrics_folder=sys_dir)
        text, tokens, total = transcribe(model, m, name, c.audio, c.duration)
        metrics_monitor_stop()
        peak = max(peak, mx.get_peak_memory() / 1e9)
        n_ref, n_err = word_errors(c.reference, text)
        ref_words, errors = ref_words + n_ref, errors + n_err
        (out_dir / f'{name}_transcript_{c.id}.txt').write_text(text + '\n')
        rows.append({
            'clip_id': c.id, 'model': name, 'family': m.family, 'size': m['size'], 'voice': c.voice,
            'input_audio (s)': round(c.duration, 2),
            'input_tokens': math.ceil(c.duration / 30) * 1500 if m.library == 'whisper' else math.ceil(c.duration * 100 / 8),
            'output_tokens': len(tokens), 'output_words': len(text.split()),
            'first_token': json.dumps(tokens[0] if tokens else '')[1:-1],   # escaped, so leading spaces stay visible
            'time_to_first_token (ms)': round(1000 * total, 1),               # not streamed: equals inference time
            'inference_rate (tokens/s)': round(len(tokens) / total, 1),
            'inference_time (s)': round(total, 3),
            'transcript': text,
        })
        print(f'  [{c.id}] {c.duration:5.1f} s audio -> {total:.2f} s, {n_err}/{n_ref} word errors  ({c.voice})')
    del model
    ModelHolder.model, ModelHolder.model_path = None, None
    gc.collect()
    mx.clear_cache()

    df = pd.DataFrame(rows)
    df.to_csv(Path(model_dir) / f'{name}_clip_metrics.csv', index=False)
    hw = pd.concat([pd.read_csv(Path(sys_dir) / f'{i}_system_metrics.csv') for i in df.clip_id])
    audio_s, time_s = df['input_audio (s)'].sum(), df['inference_time (s)'].sum()
    summary = {
        'model': name, 'family': m.family, 'size': m['size'], 'parameters': m.params, 'precision': 'float16',
        'library': m.library, 'path': m.path, 'source': m.source, 'audio_set': AUDIO_SET, 'clips': len(df),
        'model_load (s)': round(load_s, 1),
        'input_audio (s, total)': round(audio_s, 1),
        'output_tokens (total)': int(df.output_tokens.sum()),
        'word_error_rate (%)': round(100 * errors / ref_words, 2),
        'time_to_first_token (ms, avg)': round(df['time_to_first_token (ms)'].mean(), 1),
        'inference_rate (tokens/s, avg)': round(df['inference_rate (tokens/s)'].mean(), 1),
        'inference_time (s, total)': round(time_s, 2),
        'real_time_factor': round(time_s / audio_s, 4),
        'times_faster_than_real_time': round(audio_s / time_s, 1),
        'peak_memory (GB)': round(peak, 2),
        'gpu_utilization (%, avg)': round(hw['GPU Utilization (%)'].mean(), 1),
        'gpu_power (W, avg)': round(hw['gpuPower'].mean(), 1),
        'gpu_temperature (C, max)': round(hw['GPU_Temperature(C)'].max(), 1),
        'cpu_utilization (%, avg)': round(hw['CPU_Util(%)'].mean(), 1),
        'cpu_power (W, avg)': round(hw['cpuPower'].mean(), 1),
        'npu_power (W, avg)': round(hw['npuPower'].mean(), 2),
        'system_energy (J)': round(hw['total_system_power'].sum() * 0.1, 1),   # 100 ms samples
    }
    (out_dir / f'{name}_simple_metrics.json').write_text(json.dumps(summary, indent=2))
    return summary, df

summary, per_clip = run_model(MODEL)
display(per_clip.drop(columns=['transcript']))
pd.Series(summary, name=MODEL)

## 5. Compare the family

Runs every other model of `FAMILY`, each in its own fresh process (this notebook, run for that model), so every model
starts from the same clean GPU state. Set `STT_COMPARE=0` (or `COMPARE_FAMILY = False` in section 1) to skip.

In [ ]:
results = {MODEL: summary}
if COMPARE_FAMILY:
    for name, m in MODELS[MODELS.family == FAMILY].iterrows():
        if name == MODEL:
            continue
        print(f'{name}: running in a fresh process...')
        env = {**os.environ, 'STT_FAMILY': m.family, 'STT_SIZE': m['size'], 'STT_AUDIO': AUDIO_SET, 'STT_COMPARE': '0'}
        with tempfile.TemporaryDirectory() as tmp:
            run = subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'notebook', '--execute',
                                  'notebooks/stt_speech_to_text.ipynb', '--ExecutePreprocessor.timeout=-1',
                                  '--output-dir', tmp], env=env, capture_output=True, text=True)
        if run.returncode != 0:
            print(f'  failed:\n{run.stderr[-2000:]}')
            continue
        results[name] = json.loads((Path('outputs/stt') / m.family / m.params / AUDIO_SET /
                                    f'{name}_simple_metrics.json').read_text())
        print(f"  WER {results[name]['word_error_rate (%)']}%, {results[name]['times_faster_than_real_time']}x real time")
pd.DataFrame({name: results[name] for name in MODELS.index if name in results})

## 6. All models run so far

Combines every float16 model run on `AUDIO_SET` into `outputs/stt/all_models_<audio set>.csv` and their per-clip
metrics into `metrics_analysis/stt/all_models_clip_metrics_<audio set>.csv`.

In [ ]:
summaries = [json.loads(f.read_text()) for f in Path('outputs/stt').glob(f'*/*/{AUDIO_SET}/*_simple_metrics.json')]
summaries = sorted((s for s in summaries if 'path' in s),   # results from this notebook (not the earlier one)
                    key=lambda s: list(MODELS.index).index(s['model']))
all_models = pd.DataFrame(summaries)
all_models.to_csv(Path('outputs/stt') / f'all_models_{AUDIO_SET}.csv', index=False)
all_clips = pd.concat([
    pd.read_csv(Path('metrics_analysis/stt') / s['family'] / s['parameters'] / AUDIO_SET / 'Model_Level_Metrics' /
                f"{s['model']}_clip_metrics.csv", dtype={'clip_id': str})
    for s in summaries])
all_clips.to_csv(Path('metrics_analysis/stt') / f'all_models_clip_metrics_{AUDIO_SET}.csv', index=False)
print(f'{len(all_models)} models, {len(all_clips)} clip rows')
all_models